In [ ]:
!pip install fastapi uvicorn pyngrok mediapipe opencv-python-headless pillow numpy
!pip install python-multipart

In [ ]:
!ngrok authtoken 2wQfUkrrl7yd2OHI85UDUkM8igS_7XDTW1WQJ4spbDuq5QJFg

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import os

# 디렉토리 생성
os.makedirs('www', exist_ok=True)
os.makedirs('uploads', exist_ok=True)
os.makedirs('registered_faces', exist_ok=True)

In [ ]:
%%writefile www/index.html
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>안면인식 시스템</title>
    <script crossorigin src="https://unpkg.com/react@18/umd/react.production.min.js"></script>
    <script crossorigin src="https://unpkg.com/react-dom@18/umd/react-dom.production.min.js"></script>
    <script src="https://unpkg.com/@babel/standalone/babel.min.js"></script>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background-color: #f5f5f5;
        }
        .container {
            max-width: 800px;
            margin: 0 auto;
            background: white;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }
        .camera-container {
            position: relative;
            margin: 20px 0;
            text-align: center;
        }
        video {
            width: 100%;
            max-width: 640px;
            border: 2px solid #ddd;
            border-radius: 8px;
        }
        canvas {
            position: absolute;
            top: 0;
            left: 50%;
            transform: translateX(-50%);
            pointer-events: none;
        }
        .controls {
            display: flex;
            flex-wrap: wrap;
            gap: 10px;
            justify-content: center;
            margin: 20px 0;
        }
        button {
            padding: 10px 20px;
            font-size: 16px;
            cursor: pointer;
            background-color: #4CAF50;
            color: white;
            border: none;
            border-radius: 5px;
            transition: background-color 0.3s;
        }
        button:hover {
            background-color: #45a049;
        }
        button:disabled {
            background-color: #ccc;
            cursor: not-allowed;
        }
        .info-section {
            margin: 20px 0;
            padding: 15px;
            background-color: #f9f9f9;
            border-radius: 5px;
        }
        .error {
            color: red;
            margin: 10px 0;
        }
        .success {
            color: green;
            margin: 10px 0;
        }
        .form-group {
            margin: 10px 0;
        }
        label {
            display: block;
            margin-bottom: 5px;
            font-weight: bold;
        }
        input[type="text"] {
            width: 100%;
            padding: 8px;
            border: 1px solid #ddd;
            border-radius: 4px;
        }
        .recognition-results {
            margin-top: 20px;
            padding: 15px;
            background-color: #e8f5e9;
            border-radius: 5px;
        }
    </style>
</head>
<body>
    <div id="root"></div>
    <script type="text/babel">
        const { useState, useRef, useEffect } = React;

        function App() {
            const [cameraActive, setCameraActive] = useState(false);
            const [selectedCamera, setSelectedCamera] = useState('user');
            const [devices, setDevices] = useState([]);
            const [message, setMessage] = useState('');
            const [messageType, setMessageType] = useState('');
            const [isRegistering, setIsRegistering] = useState(false);
            const [registerName, setRegisterName] = useState('');
            const [recognitionResults, setRecognitionResults] = useState(null);
            const [faceRectangle, setFaceRectangle] = useState(null);

            const videoRef = useRef(null);
            const canvasRef = useRef(null);
            const streamRef = useRef(null);

            useEffect(() => {
                // 카메라 장치 목록 가져오기
                navigator.mediaDevices.enumerateDevices()
                    .then(devices => {
                        const videoDevices = devices.filter(device => device.kind === 'videoinput');
                        setDevices(videoDevices);
                    });
            }, []);

            const startCamera = async () => {
                try {
                    if (streamRef.current) {
                        streamRef.current.getTracks().forEach(track => track.stop());
                    }

                    const constraints = {
                        video: { facingMode: selectedCamera }
                    };

                    const stream = await navigator.mediaDevices.getUserMedia(constraints);
                    streamRef.current = stream;

                    if (videoRef.current) {
                        videoRef.current.srcObject = stream;
                        setCameraActive(true);
                    }
                } catch (error) {
                    showMessage('카메라를 시작할 수 없습니다: ' + error.message, 'error');
                }
            };

            const stopCamera = () => {
                if (streamRef.current) {
                    streamRef.current.getTracks().forEach(track => track.stop());
                    streamRef.current = null;
                }
                if (videoRef.current) {
                    videoRef.current.srcObject = null;
                }
                setCameraActive(false);
                setFaceRectangle(null);
            };

            const captureImage = () => {
                if (!videoRef.current) return null;

                const canvas = document.createElement('canvas');
                canvas.width = videoRef.current.videoWidth;
                canvas.height = videoRef.current.videoHeight;
                const ctx = canvas.getContext('2d');
                ctx.drawImage(videoRef.current, 0, 0);

                return canvas.toDataURL('image/jpeg');
            };

            const showMessage = (msg, type) => {
                setMessage(msg);
                setMessageType(type);
                setTimeout(() => {
                    setMessage('');
                    setMessageType('');
                }, 5000);
            };

            const registerFace = async () => {
                if (!registerName.trim()) {
                    showMessage('이름을 입력해주세요.', 'error');
                    return;
                }

                const imageData = captureImage();
                if (!imageData) {
                    showMessage('카메라가 활성화되지 않았습니다.', 'error');
                    return;
                }

                try {
                    const formData = new FormData();
                    const blob = await fetch(imageData).then(r => r.blob());
                    formData.append('image', blob, 'face.jpg');
                    formData.append('name', registerName);

                    const response = await fetch('./register', {
                        method: 'POST',
                        body: formData
                    });

                    const data = await response.json();

                    if (response.ok) {
                        showMessage(data.message, 'success');
                        setRegisterName('');
                        setIsRegistering(false);
                    } else {
                        showMessage(data.detail || '등록 실패', 'error');
                    }
                } catch (error) {
                    showMessage('등록 중 오류 발생: ' + error.message, 'error');
                }
            };

            const recognizeFace = async () => {
                const imageData = captureImage();
                if (!imageData) {
                    showMessage('카메라가 활성화되지 않았습니다.', 'error');
                    return;
                }

                try {
                    const formData = new FormData();
                    const blob = await fetch(imageData).then(r => r.blob());
                    formData.append('image', blob, 'face.jpg');

                    const response = await fetch('./recognize', {
                        method: 'POST',
                        body: formData
                    });

                    const data = await response.json();

                    if (response.ok) {
                        setRecognitionResults(data);
                        if (data.face_detected && videoRef.current) {
                            const videoWidth = videoRef.current.videoWidth;
                            const videoHeight = videoRef.current.videoHeight;
                            const displayWidth = videoRef.current.offsetWidth;
                            const displayHeight = videoRef.current.offsetHeight;

                            const scaleX = displayWidth / videoWidth;
                            const scaleY = displayHeight / videoHeight;

                            const rect = data.face_location;
                            setFaceRectangle({
                                left: rect.left * scaleX,
                                top: rect.top * scaleY,
                                width: rect.width * scaleX,
                                height: rect.height * scaleY
                            });
                        }
                        showMessage('인식 완료', 'success');
                    } else {
                        showMessage(data.detail || '인식 실패', 'error');
                    }
                } catch (error) {
                    showMessage('인식 중 오류 발생: ' + error.message, 'error');
                }
            };

            useEffect(() => {
                if (faceRectangle && canvasRef.current && videoRef.current) {
                    const canvas = canvasRef.current;
                    const ctx = canvas.getContext('2d');

                    canvas.width = videoRef.current.offsetWidth;
                    canvas.height = videoRef.current.offsetHeight;

                    ctx.clearRect(0, 0, canvas.width, canvas.height);
                    ctx.strokeStyle = '#00ff00';
                    ctx.lineWidth = 3;
                    ctx.strokeRect(
                        faceRectangle.left,
                        faceRectangle.top,
                        faceRectangle.width,
                        faceRectangle.height
                    );
                }
            }, [faceRectangle]);

            return (
                <div className="container">
                    <h1>안면인식 시스템</h1>

                    <div className="controls">
                        <select
                            value={selectedCamera}
                            onChange={(e) => setSelectedCamera(e.target.value)}
                            disabled={cameraActive}
                        >
                            <option value="user">전면 카메라</option>
                            <option value="environment">후면 카메라</option>
                        </select>

                        {!cameraActive ? (
                            <button onClick={startCamera}>카메라 시작</button>
                        ) : (
                            <button onClick={stopCamera}>카메라 중지</button>
                        )}
                    </div>

                    <div className="camera-container">
                        <video
                            ref={videoRef}
                            autoPlay
                            playsInline
                            style={{ display: cameraActive ? 'block' : 'none' }}
                        />
                        <canvas
                            ref={canvasRef}
                            style={{ display: cameraActive ? 'block' : 'none' }}
                        />
                    </div>

                    {message && (
                        <div className={messageType === 'error' ? 'error' : 'success'}>
                            {message}
                        </div>
                    )}

                    {cameraActive && (
                        <div className="controls">
                            {!isRegistering ? (
                                <>
                                    <button onClick={() => setIsRegistering(true)}>안면 등록</button>
                                    <button onClick={recognizeFace}>안면 인식</button>
                                </>
                            ) : (
                                <div className="form-group">
                                    <label>등록할 이름:</label>
                                    <input
                                        type="text"
                                        value={registerName}
                                        onChange={(e) => setRegisterName(e.target.value)}
                                        placeholder="이름을 입력하세요"
                                    />
                                    <div className="controls" style={{ marginTop: '10px' }}>
                                        <button onClick={registerFace}>등록 완료</button>
                                        <button onClick={() => {
                                            setIsRegistering(false);
                                            setRegisterName('');
                                        }}>취소</button>
                                    </div>
                                </div>
                            )}
                        </div>
                    )}

                    {recognitionResults && (
                        <div className="recognition-results">
                            <h3>인식 결과</h3>
                            <p>얼굴 감지: {recognitionResults.face_detected ? '예' : '아니오'}</p>
                            {recognitionResults.recognized && (
                                <>
                                    <p>인식된 사람: {recognitionResults.name}</p>
                                    <p>신뢰도: {(recognitionResults.confidence * 100).toFixed(2)}%</p>
                                </>
                            )}
                            {recognitionResults.face_detected && (
                                <>
                                    <p>얼굴 특징점 수: {recognitionResults.num_landmarks}</p>
                                    <p>얼굴 위치:
                                        상단-{recognitionResults.face_location.top},
                                        좌측-{recognitionResults.face_location.left},
                                        너비-{recognitionResults.face_location.width},
                                        높이-{recognitionResults.face_location.height}
                                    </p>
                                </>
                            )}
                        </div>
                    )}
                </div>
            );
        }

        ReactDOM.render(<App />, document.getElementById('root'));
    </script>
</body>
</html>

Writing www/index.html


In [ ]:
%%writefile app.py
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
import cv2
import mediapipe as mp
import numpy as np
import json
import os
import base64
from pathlib import Path
import pickle
from typing import Optional
import io
from PIL import Image

app = FastAPI()

# CORS 설정
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# MediaPipe 초기화
mp_face_detection = mp.solutions.face_detection
mp_face_mesh = mp.solutions.face_mesh
face_detection = mp_face_detection.FaceDetection(min_detection_confidence=0.5)
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.5)

# 등록된 얼굴 데이터 저장 경로
REGISTERED_FACES_DIR = Path("registered_faces")
REGISTERED_FACES_DIR.mkdir(exist_ok=True)

def extract_face_embedding(image):
    """MediaPipe를 사용한 얼굴 특징 추출"""
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_image)

    if results.multi_face_landmarks:
        face_landmarks = results.multi_face_landmarks[0]

        # 랜드마크를 numpy 배열로 변환
        landmarks = []
        for landmark in face_landmarks.landmark:
            landmarks.extend([landmark.x, landmark.y, landmark.z])

        return np.array(landmarks)

    return None

def compare_embeddings(embedding1, embedding2):
    """두 임베딩 간의 유사도 계산"""
    if embedding1 is None or embedding2 is None:
        return 0.0

    # 코사인 유사도 계산
    dot_product = np.dot(embedding1, embedding2)
    norm1 = np.linalg.norm(embedding1)
    norm2 = np.linalg.norm(embedding2)

    if norm1 == 0 or norm2 == 0:
        return 0.0

    similarity = dot_product / (norm1 * norm2)
    return similarity

def get_face_detection_info(image):
    """얼굴 감지 정보 추출"""
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_detection.process(rgb_image)

    if results.detections:
        detection = results.detections[0]
        bbox = detection.location_data.relative_bounding_box
        h, w, _ = image.shape

        return {
            "face_detected": True,
            "face_location": {
                "left": int(bbox.xmin * w),
                "top": int(bbox.ymin * h),
                "width": int(bbox.width * w),
                "height": int(bbox.height * h)
            }
        }

    return {"face_detected": False}

@app.post("/register")
async def register_face(image: UploadFile = File(...), name: str = Form(...)):
    """얼굴 등록 API"""
    try:
        # 이름 검증
        if not name.strip():
            raise HTTPException(status_code=400, detail="이름을 입력해주세요.")

        # 이미지 읽기
        contents = await image.read()
        nparr = np.frombuffer(contents, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # 얼굴 특징 추출
        embedding = extract_face_embedding(img)
        if embedding is None:
            raise HTTPException(status_code=400, detail="얼굴을 찾을 수 없습니다.")

        # 기존 등록된 얼굴과 비교
        for file in REGISTERED_FACES_DIR.glob("*.pkl"):
            with open(file, "rb") as f:
                registered_data = pickle.load(f)
                registered_embedding = registered_data["embedding"]
                similarity = compare_embeddings(embedding, registered_embedding)

                if similarity > 0.95:  # 95% 이상 유사도
                    raise HTTPException(
                        status_code=400,
                        detail=f"이미 등록된 얼굴입니다. (유사도: {similarity*100:.1f}%)"
                    )

        # 얼굴 데이터 저장
        face_data = {
            "name": name,
            "embedding": embedding
        }

        filename = f"{name}_{len(list(REGISTERED_FACES_DIR.glob('*.pkl')))}.pkl"
        with open(REGISTERED_FACES_DIR / filename, "wb") as f:
            pickle.dump(face_data, f)

        return {"message": f"{name}님의 얼굴이 성공적으로 등록되었습니다."}

    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/recognize")
async def recognize_face(image: UploadFile = File(...)):
    """얼굴 인식 API"""
    try:
        # 이미지 읽기
        contents = await image.read()
        nparr = np.frombuffer(contents, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # 얼굴 감지 정보
        detection_info = get_face_detection_info(img)

        # 얼굴 특징 추출
        embedding = extract_face_embedding(img)
        if embedding is None:
            return {
                **detection_info,
                "recognized": False,
                "message": "얼굴을 찾을 수 없습니다."
            }

        # 등록된 얼굴과 비교
        best_match = None
        best_similarity = 0

        for file in REGISTERED_FACES_DIR.glob("*.pkl"):
            with open(file, "rb") as f:
                registered_data = pickle.load(f)
                registered_embedding = registered_data["embedding"]
                similarity = compare_embeddings(embedding, registered_embedding)

                if similarity > best_similarity:
                    best_similarity = similarity
                    best_match = registered_data["name"]

        # 결과 반환
        if best_similarity > 0.85:  # 85% 이상 유사도로 인식
            # 얼굴 메시 정보 추가
            results = face_mesh.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            num_landmarks = len(results.multi_face_landmarks[0].landmark) if results.multi_face_landmarks else 0

            return {
                **detection_info,
                "recognized": True,
                "name": best_match,
                "confidence": float(best_similarity),
                "num_landmarks": num_landmarks,
                "message": f"{best_match}님으로 인식되었습니다."
            }
        else:
            return {
                **detection_info,
                "recognized": False,
                "confidence": float(best_similarity),
                "message": "등록되지 않은 얼굴입니다."
            }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health_check():
    """헬스체크 엔드포인트"""
    return {"status": "healthy"}

# 메인 페이지 라우트
@app.get("/")
async def read_index():
    return FileResponse('www/index.html')

# Static 파일 설정 - API 엔드포인트 정의 후에 마운트
app.mount("/static", StaticFiles(directory="www"), name="static")

Writing app.py


In [ ]:
%%writefile run_server.py
import subprocess
import time
from pyngrok import ngrok
import uvicorn
import os

# ngrok 터널 생성
http_tunnel = ngrok.connect(8000)
print(f"ngrok 터널이 생성되었습니다: {http_tunnel.public_url}")

# 환경 변수로 공개 URL 전달 (선택사항)
os.environ['PUBLIC_URL'] = http_tunnel.public_url

try:
    # FastAPI 서버 실행
    config = uvicorn.Config(
        "app:app",
        host="0.0.0.0",
        port=8000,
        reload=False,
        log_level="info"
    )
    server = uvicorn.Server(config)
    server.run()
except KeyboardInterrupt:
    # 종료 시 정리
    ngrok.kill()
    print("서버가 종료되었습니다.")

Writing run_server.py


In [ ]:
!python run_server.py

ngrok 터널이 생성되었습니다: https://06b6-35-240-252-50.ngrok-free.app
2025-05-14 05:45:26.113850: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747201526.134641    1869 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747201526.140535    1869 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-14 05:45:26.160010: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
